# GSK Pharma Pipeline Analysis

Analysing clinical trial performance by therapeutic area with adverse event profiling.

**Data source:** `TH_DEMO_DB.HOL_2_PHARMA`

In [ ]:
%%sql -r dataframe_1
-- Cell 1: Setup context
USE DATABASE TH_DEMO_DBBB;
USE SCHEMA HOL_2_PHARMA;
USE WAREHOUSE DEMO_WH;

In [ ]:
# Cell 2: Trial performance by therapeutic area
import snowflake.snowpark as snowpark
from snowflake.snowpark.functions import col, count, sum as sum_
import pandas as pd

trials_df = session.table('CLINICAL_TRIALS')
compounds_df = session.table('COMPOUNDS')

joined = trials_df.join(compounds_df, trials_df['compound_key'] == compounds_df['compound_id'])

result = joined.group_by('THERAPEUTIC_AREA').agg(
    count('TRIAL_ID').alias('TOTAL_TRIALS'),
    sum_(col('SUCCESS').cast('integer')).alias('SUCCESSFUL_TRIALS')
).to_pandas()

result['SUCCESS_RATE'] = result['SUCCESSFUL_TRIALS'] / result['TOTAL_TRIALS'] * 100
result.sort_values('SUCCESS_RATE', ascending=False)

In [ ]:
# Cell 3: Adverse event summary chart
ae_df = session.table('ADVERSE_EVENTS').to_pandas()

severity_counts = ae_df.groupby(['EVENT_TYPE', 'SEVERITY']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
severity_counts.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Adverse Events by Type and Severity')
ax.set_xlabel('Event Type')
ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Risk score per compound
ae_data = session.table('ADVERSE_EVENTS').to_pandas()
trials_data = session.table('CLINICAL_TRIALS').to_pandas()
compounds_data = session.table('COMPOUNDS').to_pandas()

ae_data['severity_score'] = ae_data['SEVERITY'].astype(int)

risk_by_trial = ae_data.groupby('TRIAL_ID').agg(
    total_risk=('severity_score', 'sum'),
    event_count=('EVENT_ID', 'count')
).reset_index()

risk_with_info = risk_by_trial.merge(
    trials_data[['TRIAL_ID', 'ENROLLED_PATIENTS', 'COMPOUND_ID']], on='TRIAL_ID'
).merge(
    compounds_data[['COMPOUND_ID', 'COMPOUND_NAME', 'THERAPEUTIC_AREA']], on='COMPOUND_ID'
)

risk_with_info['risk_per_patient'] = risk_with_info['total_risk'] / risk_with_info['ENROLLED_PATIENTS']
risk_with_info[['COMPOUND_NAME', 'THERAPEUTIC_AREA', 'event_count', 'risk_per_patient']].sort_values('risk_per_patient', ascending=False).head(10)